# Member 1 - Prepare data for AI

## 1. Load Dataset

In [ ]:
!pip install pandas

In [15]:
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1', header=0, names=['label', 'message', 'ex1', 'ex2', 'ex3'])

# Check the dimensions (Rows, Columns)
print("Original Dataset:")
print(f"Dataset: {df.shape}")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}\n")

# Clean dataset
df = df.iloc[:, :2] # keep only label and message
df.columns = ['label', 'message']

print("Data loaded successfully!\n")
print("Updated Dataset:")
print(f"Dataset Shape: {df.shape}")
print(f"Total Rows: {df.shape[0]}\n")
print(df.head())

Original Dataset:
Dataset: (5572, 5)
Total Rows: 5572
Total Columns: 5

Data loaded successfully!

Updated Dataset:
Dataset Shape: (5572, 2)
Total Rows: 5572

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


## 2. Data Cleaning

In [16]:
# Inspect before cleaning
print("Values in label column before mapping:")
print(df['label'].unique())

# Remove duplicates and missing values
df = df.drop_duplicates(keep='first')
df = df.dropna()

# Label Encoding
df['label'] = df['label'].str.strip().map({'ham': 0, 'spam': 1})

# 4. Check again
print("\nMissing values after cleaning:")
print(df.isnull().sum())
print("\nClass distribution (0=Ham, 1=Spam):")
print(df['label'].value_counts())

Values in label column before mapping:
['ham' 'spam']

Missing values after cleaning:
label      0
message    0
dtype: int64

Class distribution (0=Ham, 1=Spam):
label
0    4516
1     653
Name: count, dtype: int64


## 3. Text Preprocessing

In [19]:
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower() # Lowercase
    text = text.translate(str.maketrans('', '', string.punctuation)) # Remove punctuation
    words = [w for w in text.split() if w not in stop_words] # Remove stopwords
    return " ".join(words)

df['clean_message'] = df['message'].apply(preprocess)
print("Preprocessing complete!")
print("\nSample of clean_message:")
print(df[['message', 'clean_message']].head())

Preprocessing complete!

Sample of clean_message:
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry 2 wkly comp win fa cup final tkts 2...  
3                u dun say early hor u c already say  
4        nah dont think goes usf lives around though  


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 4. Feature Extraction

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['clean_message'])

print("TF-IDF Vectorization complete!")
print(f"Matrix shape: {X.shape}")

TF-IDF Vectorization complete!
Matrix shape: (5169, 3000)


## 5. Export Files

In [26]:
# Export to CSV
df.to_csv('cleaned_spam_data.csv', index=False)

# Export the vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("Files saved!")

Files saved!


# Validation (Testing)

### Text preprocessing

In [24]:
sample_text = "Win a FREE iPhone now!!!, call 555-0199."
cleaned_sample = preprocess(sample_text)

print(f"Original: {sample_text}")
print(f"\nCleaned:  {cleaned_sample}")

Original: Win a FREE iPhone now!!!, call 555-0199.

Cleaned:  win free iphone call 5550199


### TF-IDF vectorizer

In [25]:
# Transform the cleaned sample
vector = tfidf.transform([cleaned_sample])
print(f"Vector shape: {vector.shape}")

Vector shape: (1, 3000)
